In [1]:
# იმპორტები
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd  
import numpy as np 
import os, re
import string
import nltk 
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import nltk
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud,STOPWORDS
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense
from nltk.corpus import stopwords
import torch

%matplotlib inline

In [2]:
#ვქმნი ფაილის path სტრინგებს: 
file_path_labeledTrainData = r"C:\Users\LG\Downloads\NLP_HW2\labeledTrainData.tsv"
file_path_testData = r"C:\Users\LG\Downloads\NLP_HW2\testData.tsv"

#ვკითხულობ ფაილებს
df_labeledTrainData = pd.read_csv(file_path_labeledTrainData, delimiter='\t')
df_testData = pd.read_csv(file_path_testData, delimiter='\t')

#ვბეჭდავ ფაილებს
print(df_labeledTrainData.head())
print(df_testData.head())

       id  sentiment                                             review
0  5814_8          1  With all this stuff going down at the moment w...
1  2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2  7759_3          0  The film starts with a manager (Nicholas Bell)...
3  3630_4          0  It must be assumed that those who praised this...
4  9495_8          1  Superbly trashy and wondrously unpretentious 8...
         id                                             review
0  12311_10  Naturally in a film who's main themes are of m...
1    8348_2  This movie is a disaster within a disaster fil...
2    5828_4  All in all, this is a movie for kids. We saw i...
3    7186_2  Afraid of the Dark left me with the impression...
4   12128_7  A very accurate depiction of small time mob li...


In [3]:
#ვიგებ მონეცემების ფორმატის შესახებ ინფორმაციას
df_labeledTrainData.info()
df_labeledTrainData.describe()
df_labeledTrainData['review'].isnull().values.any()

#ვბეჭდავ მონაცემებს უფრო ვრცელ ფორმატში
pd.set_option('display.max_colwidth', None)
print(df_labeledTrainData['review'])
print(df_labeledTrainData['sentiment'])
print(df_testData['review'])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
 2   review     25000 non-null  object
dtypes: int64(1), object(2)
memory usage: 586.1+ KB
0                                                                                                                                                           With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has s

In [4]:
# 25000 რევიუ და სენტიმენტია რომელთაგან არცერთია NULL.
# 25000 რევიუა ტესტ მონაცემებშიც.
# დიდი ტექსტებია სადაც: HTML ტაგები ბევრია, დიდი ასოები დაპატარავებელი არ არის, სიტყვები მოკლე ფორმით არის ჩაწერილი (I'am), 
# არტიკლები და კავშირები ბევრი გვხვდება, პუნქტუაციის სიმბოლოებიც ბევრია.
# ეს ყველაფერი უნდა გაიწმინდოს


In [5]:
# ეს ფუნქცია ტექსტში html ტაგებს წმინდავს
def remove_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

#ვაშორებ HTML თაგებს
df_labeledTrainData['review'] = df_labeledTrainData['review'].apply(remove_html_tags)
df_testData['review'] = df_testData['review'].apply(remove_html_tags)

C:\Users\LG\AppData\Local\Programs\Python\Python311\Lib\site-packages\bs4\__init__.py:435: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  warnings.warn(


In [6]:
#ვბეჭდავ რათა ვნახო მოშორდა თუ არა HTML თაგები
pd.set_option('display.max_colwidth', None)
print(df_labeledTrainData['review'])
print(df_testData['review'])


0                                                                                                                                                                                               With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the maki

In [7]:
#მონაცემებში სიტყვის შემოკლებები ბევრი გვხვდება ამიტომ მათ ორ სიტყვად გადავაქცევ ამ ფუნქციით (I'am -> i am) 

def clean_text(text):
    # Replace contractions with their expanded forms
    text = re.sub(r"i'm", "i am", text, flags=re.IGNORECASE)
    text = re.sub(r"i've", "i have", text, flags=re.IGNORECASE)
    text = re.sub(r"i'll", "i will", text, flags=re.IGNORECASE)
    text = re.sub(r"i'd", "i would", text, flags=re.IGNORECASE)
    text = re.sub(r"you're", "you are", text, flags=re.IGNORECASE)
    text = re.sub(r"you've", "you have", text, flags=re.IGNORECASE)
    text = re.sub(r"you'll", "you will", text, flags=re.IGNORECASE)
    text = re.sub(r"you'd", "you would", text, flags=re.IGNORECASE)
    text = re.sub(r"he's", "he is", text, flags=re.IGNORECASE)
    text = re.sub(r"he'll", "he will", text, flags=re.IGNORECASE)
    text = re.sub(r"he'd", "he would", text, flags=re.IGNORECASE)
    text = re.sub(r"she's", "she is", text, flags=re.IGNORECASE)
    text = re.sub(r"she'll", "she will", text, flags=re.IGNORECASE)
    text = re.sub(r"she'd", "she would", text, flags=re.IGNORECASE)
    text = re.sub(r"it's", "it is", text, flags=re.IGNORECASE)
    text = re.sub(r"it'll", "it will", text, flags=re.IGNORECASE)
    text = re.sub(r"it'd", "it would", text, flags=re.IGNORECASE)
    text = re.sub(r"we're", "we are", text, flags=re.IGNORECASE)
    text = re.sub(r"we've", "we have", text, flags=re.IGNORECASE)
    text = re.sub(r"we'll", "we will", text, flags=re.IGNORECASE)
    text = re.sub(r"we'd", "we would", text, flags=re.IGNORECASE)
    text = re.sub(r"they're", "they are", text, flags=re.IGNORECASE)
    text = re.sub(r"they've", "they have", text, flags=re.IGNORECASE)
    text = re.sub(r"they'll", "they will", text, flags=re.IGNORECASE)
    text = re.sub(r"they'd", "they would", text, flags=re.IGNORECASE)
    text = re.sub(r"that's", "that is", text, flags=re.IGNORECASE)
    text = re.sub(r"that'll", "that will", text, flags=re.IGNORECASE)
    text = re.sub(r"that'd", "that would", text, flags=re.IGNORECASE)
    text = re.sub(r"what's", "what is", text, flags=re.IGNORECASE)
    text = re.sub(r"what'll", "what will", text, flags=re.IGNORECASE)
    text = re.sub(r"what'd", "what would", text, flags=re.IGNORECASE)
    text = re.sub(r"where's", "where is", text, flags=re.IGNORECASE)
    text = re.sub(r"where'll", "where will", text, flags=re.IGNORECASE)
    text = re.sub(r"where'd", "where would", text, flags=re.IGNORECASE)
    text = re.sub(r"who's", "who is", text, flags=re.IGNORECASE)
    text = re.sub(r"who'll", "who will", text, flags=re.IGNORECASE)
    text = re.sub(r"who'd", "who would", text, flags=re.IGNORECASE)
    text = re.sub(r"how's", "how is", text, flags=re.IGNORECASE)
    text = re.sub(r"how'll", "how will", text, flags=re.IGNORECASE)
    text = re.sub(r"how'd", "how did", text, flags=re.IGNORECASE)
    text = re.sub(r"there's", "there is", text, flags=re.IGNORECASE)
    text = re.sub(r"there'll", "there will", text, flags=re.IGNORECASE)
    text = re.sub(r"there'd", "there would", text, flags=re.IGNORECASE)
    text = re.sub(r"that'll", "that will", text, flags=re.IGNORECASE)
    text = re.sub(r"who've", "who have", text, flags=re.IGNORECASE)
    text = re.sub(r"let's", "let us", text, flags=re.IGNORECASE)
    text = re.sub(r"won't", "will not", text, flags=re.IGNORECASE)
    text = re.sub(r"can't", "cannot", text, flags=re.IGNORECASE)
    text = re.sub(r"shan't", "shall not", text, flags=re.IGNORECASE)
    text = re.sub(r"shouldn't", "should not", text, flags=re.IGNORECASE)
    text = re.sub(r"wouldn't", "would not", text, flags=re.IGNORECASE)
    text = re.sub(r"didn't", "did not", text, flags=re.IGNORECASE)
    text = re.sub(r"doesn't", "does not", text, flags=re.IGNORECASE)
    text = re.sub(r"wasn't", "was not", text, flags=re.IGNORECASE)
    text = re.sub(r"weren't", "were not", text, flags=re.IGNORECASE)
    text = re.sub(r"haven't", "have not", text, flags=re.IGNORECASE)
    text = re.sub(r"hasn't", "has not", text, flags=re.IGNORECASE)
    text = re.sub(r"aren't", "are not", text, flags=re.IGNORECASE)
    text = re.sub(r"ain't", "am not", text, flags=re.IGNORECASE)
    return text


In [8]:
#შემოკლებუს სიტყვებს ორად ვხლეჩავ მონაცემებში
df_labeledTrainData['review'] = df_labeledTrainData['review'].apply(lambda text: clean_text(text))
df_testData['review'] = df_testData['review'].apply(lambda text: clean_text(text))     


In [9]:
#ვბეჭდავ რათა ვნახო გასწორდა თუ არა შემოკლებული სიტყვები
pd.set_option('display.max_colwidth', None)
print(df_labeledTrainData['review'])
print(df_testData['review'])

0                                                                                                                                                                                           With all this stuff going down at the moment with MJ i have started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making

In [10]:
#ეს ფუნქცია აკეთებს ტექსტის ლემატიზაციას ანუ სიტყვებს ბუნებრივ ფორმაში აბრუნებს 
# (I was going to school while eating) -> (i am go to school while eat)
def lemmatize_text(text):
    words = word_tokenize(text)
    lemmatized_words = []
    for word, tag in nltk.pos_tag(words):
        wn_tag = tag[0].lower()
        wn_tag = wn_tag if wn_tag in ['a', 'r', 'n', 'v'] else None
        if not wn_tag:
            wn_tag = 'n'
        lemma = lemmatizer.lemmatize(word, pos=wn_tag)
        lemmatized_words.append(lemma)
    lemmatized_text = ' '.join(lemmatized_words)
    return lemmatized_text


In [11]:
# ვუკეთებ მონაცემებს ლემატიზაციას

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df_labeledTrainData['review'] = df_labeledTrainData['review'].apply(lambda text: lemmatize_text(text))
df_testData['review'] = df_testData['review'].apply(lambda text: lemmatize_text(text))

In [12]:
#ვბეჭდავ რათა ვნახო მოხდა თუ არა ლემატიზაცია
pd.set_option('display.max_colwidth', None)
print(df_labeledTrainData['review'])
print(df_testData['review'])

0                                                                                                                                                                                                With all this stuff go down at the moment with MJ i have start listen to his music , watch the odd documentary here and there , watch The Wiz and watch Moonwalker again . Maybe i just want to get a certain insight into this guy who i think be really cool in the eighty just to maybe make up my mind whether he be guilty or innocent . Moonwalker be part biography , part feature film which i remember go to see at the cinema when it be originally release . Some of it have subtle message about MJ 's feeling towards the press and also the obvious message of drug be bad m'kay.Visually impressive but of course this be all about Michael Jackson so unless you remotely like MJ in anyway then you be go to hate this and find it bore . Some may call MJ an egotist for consent to the making of this movie BUT MJ a

In [13]:
#დიდ ასოებს ვაპატარავებ
df_labeledTrainData['review'] = df_labeledTrainData['review'].str.lower()
df_testData['review'] = df_testData['review'].str.lower()

In [14]:
#ფუნქცია ორმაგ სა მეტ space-ს ერთად გარდაქმნის და პუნქტუაციის სიმბოლოებს მოაშორებს მონაცემებს
nltk.download('punkt')

def clean_text_punct(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text

#გავწმინდე მონაცემები პუნქტუაციის სიმბოლოებისაგან
df_labeledTrainData['review'] = df_labeledTrainData['review'].apply(clean_text_punct)
df_testData['review'] = df_testData['review'].apply(clean_text_punct)

#ვბეჭდავ რათა ვნახო გაიწმინდა უ არა მონაცემები
pd.set_option('display.max_colwidth', None)
print(df_labeledTrainData['review'])
print(df_testData['review'])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LG\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


0                                                                                                                                       with all this stuff go down at the moment with mj i have start listen to his music watch the odd documentary here and there watch the wiz and watch moonwalker again maybe i just want to get a certain insight into this guy who i think be really cool in the eighty just to maybe make up my mind whether he be guilty or innocent moonwalker be part biography part feature film which i remember go to see at the cinema when it be originally release some of it have subtle message about mj s feeling towards the press and also the obvious message of drug be bad mkayvisually impressive but of course this be all about michael jackson so unless you remotely like mj in anyway then you be go to hate this and find it bore some may call mj an egotist for consent to the making of this movie but mj and most of his fan would say that he make it for the fan which if true be 

In [15]:
#ვწმინდავ სტოპსიტყვებს არტიკლები შორისდებულები კავშირები ...
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_sentence = [word for word in word_tokens if word.lower() not in stop_words]
    return ' '.join(filtered_sentence)



df_labeledTrainData['review'] = df_labeledTrainData['review'].apply(remove_stopwords)
df_testData['review'] = df_testData['review'].apply(remove_stopwords)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LG\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
#ვბეჭდავ საბოლოო გაწმენდილ მონაცემებს
pd.set_option('display.max_colwidth', None)
print(df_labeledTrainData['review'])
print(df_testData['review'])

0                                                                                                                                                                                                                                                                                                                                                                                                                   stuff go moment mj start listen music watch odd documentary watch wiz watch moonwalker maybe want get certain insight guy think really cool eighty maybe make mind whether guilty innocent moonwalker part biography part feature film remember go see cinema originally release subtle message mj feeling towards press also obvious message drug bad mkayvisually impressive course michael jackson unless remotely like mj anyway go hate find bore may call mj egotist consent making movie mj fan would say make fan true really nice himthe actual feature film bit finally start 20 minute exclude smooth cri

In [17]:
# ჯერ დავიწყოთ მარტივი ემბედინგებით
# ანუ რევიუ ტექსტის თითო სიტყვას შევუსაბამოთ 100 განზომილებიანი ვექტორი რომელიც იქნება რანდომ ვექტორი დასქეილებული სიტყვის
# ტექსტში გამოყენების სიხშირეზე

In [18]:
# მოვახდინოთ ტოკენიზაცია
tokenized_reviews = df_labeledTrainData['review'].apply(lambda x: word_tokenize(x.lower()))

tokenized_reviews_test = df_testData['review'].apply(lambda x: word_tokenize(x.lower()))

In [19]:

#დავითვალოთ სიტყვები
word_counter = Counter()
for review in tokenized_reviews:
    word_counter.update(review)

# მოვახდინოთ ინდექს მაპინგი
word_index = {word: index + 1 for index, (word, _) in enumerate(word_counter.most_common())}

max_words = len(word_index)

# მოვახდინოთ ემბედინგების რანდომიზაცია
embedding_dim = 50 
embeddings_index = {}
for word, i in word_index.items():
    embeddings_index[word] = np.random.uniform(-1, 1, embedding_dim) * (word_counter[word] / len(tokenized_reviews))
#დავასქეილოთ სიხშირის მიხედვით    

#გავაკეთოთ იგივე ტესტ დატაზე
word_counter_test = Counter()
for review in tokenized_reviews_test:
    word_counter_test.update(review)

word_index_test = {word: index + 1 for index, (word, _) in enumerate(word_counter_test.most_common())}

max_words_test = len(word_index_test)

embeddings_index_test = {}
for word, i in word_index_test.items():
        embeddings_index_test[word] = np.random.uniform(-1, 1, embedding_dim) * (word_counter_test[word] / len(tokenized_reviews_test))

In [20]:
# შევამოყმოთ ემბედინგები თუ გამოთვალა სიტყვა good - ზე ტრაინ და ტესტ მონაცემებზე
# ამ მეთოდით ხშირად გამოყენებული სიტყვებს მეტად მსავსი ემბედინგები ექნება
print(embeddings_index_test['good'])
print(embeddings_index['good'])

[-0.25700784  0.47442489  0.20797492 -0.23755053  0.4874454   0.22364399
 -0.29200263  0.46963843  0.16158108 -0.49473604 -0.35256262  0.20034557
 -0.12043243  0.00354436 -0.00787866 -0.41716083 -0.40435856  0.04162066
 -0.34637781 -0.41037993  0.06198808  0.03845468  0.05724176  0.04461582
  0.02510789 -0.21391522 -0.28762658 -0.08662639 -0.51914334 -0.11312046
 -0.43338512 -0.28753248 -0.50501876 -0.52506517  0.5415952  -0.39940299
 -0.44239306 -0.53929738 -0.07181557 -0.05998043  0.295522    0.14360809
 -0.12676796  0.00078499  0.08942978  0.23882128  0.03064721 -0.05070164
 -0.52181712  0.36434676]
[ 0.02660774 -0.07251368 -0.35098382 -0.35030319  0.53571122  0.18477219
 -0.08643867 -0.42978506 -0.37701991  0.45915533  0.55496224 -0.18926096
 -0.14340202 -0.44242353  0.48225048  0.35722123 -0.08686378  0.43947579
  0.49233367 -0.51464612  0.44385904 -0.35813259  0.17170081  0.04008402
 -0.34447048  0.38976405 -0.57258953 -0.18047831  0.20028441 -0.24923627
 -0.51619752  0.39063032 

In [21]:
embeddings = torch.tensor(list(embeddings_index.values()))
embeddings_test = torch.tensor(list(embeddings_index_test.values()))
print("Shape of embeddings:", embeddings.shape)
print("Shape of embeddings:", embeddings_test.shape)
#სულ დაითვალა 114559 ტრეინ და 111779 ტესტ სეტში
#თითო 50 განზომილებიანია

C:\Users\LG\AppData\Local\Temp\ipykernel_11020\2995801957.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ..\torch\csrc\utils\tensor_new.cpp:278.)
  embeddings = torch.tensor(list(embeddings_index.values()))


Shape of embeddings: torch.Size([114559, 50])
Shape of embeddings: torch.Size([111779, 50])


In [22]:
#მონაცემები ანუ ფილმის რევიუები გარდავქმნათ სიტყვების ემბედინგების მიმდევრობად
sequences = []
for review in df_labeledTrainData['review']:
    sequence = [embeddings_index[word] for word in review.split() if word in embeddings_index]
    sequences.append(sequence)
    
sequences_test = []
for review in df_testData['review']:
    sequence = [embeddings_index[word] for word in review.split() if word in embeddings_index]
    sequences_test.append(sequence)


In [23]:
print("Shape of input array:", len(sequences))
print("First element of input array:", sequences[0])
#ანუ 25000 რევიუდან თითოეული არის სიტყვების ემბედინგების ლისტი 

#და ეს ლისტი ემბედინგ - ემბედინგით გადაეცემა რეკურენტულ ქსელს

Shape of input array: 25000
First element of input array: [array([ 0.00932564,  0.01927084,  0.03828673,  0.02338592, -0.02232   ,
        0.0202091 , -0.01765404,  0.01069873, -0.01507502, -0.00251241,
        0.0173887 ,  0.03335264,  0.04520537, -0.03842005, -0.01751114,
       -0.03637928,  0.02590736,  0.00325366,  0.02118185,  0.04107881,
       -0.04204388, -0.04325791, -0.00968953,  0.04550473,  0.00814386,
        0.03412737,  0.03207099,  0.02297945,  0.01221479,  0.02230414,
        0.00095781, -0.0304341 ,  0.00871089,  0.04618398, -0.01006157,
       -0.03271231,  0.04076626,  0.01434796, -0.03103746,  0.01975883,
       -0.01331631,  0.0264016 , -0.01536427, -0.03729422, -0.01220243,
       -0.00368249, -0.03146525,  0.04448444,  0.03721863, -0.02854011]), array([-0.10226374,  0.31858093, -0.18443464, -0.5145249 ,  0.00566122,
       -0.3734747 , -0.4971375 ,  0.46677412,  0.25676783,  0.38590831,
        0.00217222,  0.50392876, -0.49825012,  0.00988019, -0.51797763,
   

In [24]:
#დავყოფ 80/20 ტრეინ და ვალიდაციის სეტებად

train_ratio = 0.8
val_ratio = 0.2

split_index = int(len(sequences) * train_ratio)

train_sequences = sequences[:split_index]
val_sequences = sequences[split_index:]


print("Number of sequences in training set:", len(train_sequences))
print("Number of sequences in validation set:", len(val_sequences))

Number of sequences in training set: 20000
Number of sequences in validation set: 5000


In [25]:
#გრძელი რევოუები პროცესს შეანელებს ამიტომ 300 სიტყვის მეტს აღარ მივაქცენ ყურადღებას და 300 ზე გავაკეთებ პადინგს.
maxlen = 300

x = pad_sequences(sequences, maxlen=maxlen, dtype='float32', padding='post', truncating='post')
y = np.array(df_labeledTrainData['sentiment'])

test_x = pad_sequences(sequences_test, maxlen=maxlen, dtype='float32', padding='post', truncating='post')

In [26]:
print(x.shape)
print(test_x.shape)
print(df_labeledTrainData['sentiment'].shape)
#ზომები კარგად არის

(25000, 300, 50)
(25000, 300, 50)
(25000,)


In [27]:
print(np.isnan(x).any())

False


In [28]:
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import ModelCheckpoint
#ნეირონული ქსელის იმპორტები

In [29]:
#ვაკეთებ ასე:
#ვუშვებ ინფუთ ლეიერიდან lstm ლეიერში მერე overfit რო არ მოხდეს ვუკეთებ დროფაუთს 
#და ამას ვიმეორებ 2 ჯერ გრადიენტის ვანიშინგი რო არ მოხდეს
#მერე flatten - ს ვუკეთებ და საბოლოოდ Dense ლეიერით ერთ ნოუდში მისული რიცხვს ვდებ სიგმოიდ ფუნქვიას
model = Sequential([])
model.add(layers.Input(shape=(300,50)))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.Dropout(0.1))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.Dropout(0.1))
model.add(layers.Flatten())
model.add(layers.Dense(1, activation='sigmoid'))

In [30]:
model.summary()
#მოდელის ვიზუალიზაცია

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Layer (type)                       ┃ Output Shape                  ┃     Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ lstm (LSTM)                        │ (None, 300, 80)               │      41,920 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ dropout (Dropout)                  │ (None, 300, 80)               │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ lstm_1 (LSTM)                      │ (None, 300, 80)               │      51,520 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ dropout_1 (Dropout)                │ (None, 300, 80)               │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ flatten (Flatten)                  │ (None, 24000)                 │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ dense (Dense)                      │ (None, 1)                     │      24,001 │
└────────────────────────────────────┴───────────────────────────────┴─────────────┘

 Total params: 117,441 (458.75 KB)

 Trainable params: 117,441 (458.75 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
df_labeledTrainData['sentiment'] = df_labeledTrainData['sentiment'].astype(np.float32)

#ადამის ოპტიმაიზერით ვუშვებ მოდელს 
#რადგან სენტიმენტი ან უარყოფითია ან დადებითი loss='binary_crossentropy'
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

In [32]:
y = np.array(df_labeledTrainData['sentiment'])

# ვაძლევ მოდელს x და y-ს
# ვაძლევ მოდელს ტრეინ და ტარგეტ სეტს
# ვაკეთებ validation_split=0.2, 80/20 ტრეინ და ვალიდაციის სეტს
# epochs=4 (მაინც არ ვარგა ამ მოდელის ემბედინგები და 4 - წესით ეყოფა, უკეთეს ემბედინგებს მერე გავაკეთებ)
history = model.fit(
    x=x,
    y=y,
    batch_size=50,
    epochs=4,
    verbose="auto",
    callbacks=None,
    validation_split=0.2,
    validation_data=None,
    shuffle=True,
    class_weight=None,
    sample_weight=None,
    initial_epoch=0,
    steps_per_epoch=None,
    validation_steps=None,
    validation_batch_size=None,
    validation_freq=1,
)

Epoch 1/4
400/400 ━━━━━━━━━━━━━━━━━━━━ 183s 437ms/step - accuracy: 0.5791 - loss: 0.6667 - val_accuracy: 0.6528 - val_loss: 0.6285
Epoch 2/4
400/400 ━━━━━━━━━━━━━━━━━━━━ 155s 388ms/step - accuracy: 0.6686 - loss: 0.6065 - val_accuracy: 0.6720 - val_loss: 0.6174
Epoch 3/4
400/400 ━━━━━━━━━━━━━━━━━━━━ 157s 392ms/step - accuracy: 0.6912 - loss: 0.5854 - val_accuracy: 0.6768 - val_loss: 0.5991
Epoch 4/4
400/400 ━━━━━━━━━━━━━━━━━━━━ 155s 387ms/step - accuracy: 0.6959 - loss: 0.5750 - val_accuracy: 0.6776 - val_loss: 0.5923


In [33]:
print(history.history['loss'])
print(history.history['accuracy'])
print(history.history['val_loss'])
print(history.history['val_accuracy'])

[0.6446971893310547, 0.6037004590034485, 0.5833221077919006, 0.5755364894866943]
[0.6209501028060913, 0.6715496182441711, 0.691450297832489, 0.697999894618988]
[0.6285359859466553, 0.6174459457397461, 0.5990932583808899, 0.592345118522644]
[0.6528000831604004, 0.6720000505447388, 0.6768000721931458, 0.6776000261306763]


In [34]:
# 70.85% -იანი შედეგი
# რადგან მე ემბედინგებად სიხშირეები დავასქეილე რანდომ მონაცემებზე მოდელმა უკეთ ისწავლა
# ხშირი და ნაკლებად ხშირი სიტყვების გარჩევა ამიტომ სიტყვა good და bad რომლებიც მატად ხშირად გვხვდება ემბედინგებით ახლოს
# არიან ასე რომ ეს ემბედინგები სენტიმენტის ანალიზის დროს კარგ შედეგს არ მოგვცემს

In [35]:
predictions = (model.predict(test_x, batch_size=20, verbose="auto", steps=None, callbacks=None) > 0.5).astype(int)


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 104s 82ms/step


In [36]:
# id და სენტიმენტი გავაერთიანე საბოლოო ფორმატში

In [37]:
predictions_df = pd.DataFrame(data={'id': df_testData['id'], 'sentiment': predictions.flatten()})
print(predictions_df)

             id  sentiment
0      12311_10          1
1        8348_2          0
2        5828_4          1
3        7186_2          0
4       12128_7          1
...         ...        ...
24995   2155_10          1
24996     59_10          1
24997    2531_1          1
24998    7772_8          1
24999  11465_10          1

[25000 rows x 2 columns]


In [38]:
predictions_df.to_csv('answers.csv', index = False)
# ტესტ სეტზე რო შევამოყმე იქაც 70.51 % - იანი შედეგი მომცა
# ვიცი ტესტ სეტზე მარტო ბოლოს უნდა შეამოწმო თორე ოვერფიტის საშიშროებაა
#მარა მაინტერესებდა სწორი იყო თუ არა პროცენტული დამთხვევა და ერთხელ შევამოწმე

In [40]:
# ახლა ავიღოთ მატად კომპლექსური ემბედინგები 
# ბონუს დავალების ემბედინგები გამოვითვალოთ რომელსაც გადავცემთ მოდელს

In [42]:
# word2vec ალგორითმი
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# ტექსტის დაკონეზირება
tokenized_texts = [text.split() for text in df_labeledTrainData['review']]

# სიხშირეების დათვლა
word_freq = defaultdict(int)
for text in tokenized_texts:
    for word in text:
        word_freq[word] += 1

# ლექსიკონის შექმნა სიხშირეების მიხედვით
vocab_size = 1000
sorted_word_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
vocab = [pair[0] for pair in sorted_word_freq[:vocab_size]]

# სიტყვების მაპინგის შექმნა
word_to_idx = {word: idx for idx, word in enumerate(vocab)}

# მოხვედრების მატრიცის შექმნა
window_size = 2
co_occurrence_matrix = np.zeros((vocab_size, vocab_size))
for text in tokenized_texts:
    for i, word in enumerate(text):
        if word in vocab:
            start = max(0, i - window_size)
            end = min(len(text), i + window_size + 1)
            context = text[start:end]
            context.remove(word)
            for context_word in context:
                if context_word in vocab:
                    co_occurrence_matrix[word_to_idx[word]][word_to_idx[context_word]] += 1

# ნორმალიზაცია
co_occurrence_matrix = normalize(co_occurrence_matrix, norm='l2', axis=1)

# SVD ინფორმაციის განზომილების შემცირება
embedding_dim = 50
svd = TruncatedSVD(n_components=embedding_dim)
word_vectors = svd.fit_transform(co_occurrence_matrix)



In [43]:
for word in word_to_idx:
    print(word + "    " + str(word_vectors[word_to_idx[word]]))

with open('bonus_home_workembeddings', 'w') as file:
    for word in word_to_idx:
        file.write(word + "    " + str(word_vectors[word_to_idx[word]]) + "\n")

movie    [ 8.32607218e-01 -2.02589231e-01 -1.01865484e-01 -2.66146124e-01
  2.42268853e-01  5.11906630e-02  1.22221634e-01  4.91412621e-02
  8.89997422e-02  1.17440259e-01  4.20611833e-02  1.15278389e-01
 -4.59610712e-02  9.00206169e-02 -2.94406971e-02  3.70984150e-02
  3.08520250e-03 -7.39389193e-03 -3.30452660e-02  5.72456532e-02
 -4.02947961e-02 -4.13567340e-02 -2.36264181e-02 -2.23574520e-02
 -7.52029789e-03  8.22510056e-03  1.81124075e-02  1.13424095e-02
  3.34444459e-02  1.08976043e-02 -2.72207897e-02 -6.52379471e-02
 -8.73186112e-03  3.38805971e-02 -5.51161025e-04  1.11220909e-02
 -4.86879644e-02  1.50803696e-03  3.59148881e-02 -1.25691933e-02
 -1.39056276e-02  3.21495462e-02 -3.87549425e-03  7.58068232e-03
 -3.16940020e-02 -2.60740581e-02 -3.62671256e-03 -7.48090426e-04
  2.06722770e-02 -1.56547822e-02]
film    [ 8.59125940e-01 -1.96743743e-01  3.90275836e-02 -1.39643948e-01
  2.55938437e-01 -4.93833022e-02  1.86213096e-01  8.65437359e-02
  9.55544109e-02  7.98623386e-02 -8.662

obviously    [ 0.91231682  0.1107438   0.02310699 -0.00545906  0.01401164 -0.11677064
  0.06694654  0.02767379 -0.03727361 -0.01792651  0.02699233 -0.06965873
  0.02922557 -0.08345137 -0.01823381 -0.00489455 -0.01201267  0.01717389
 -0.04674014 -0.05339003 -0.0176373  -0.02848982 -0.0097898  -0.03002442
 -0.00695051 -0.01292827  0.022824   -0.03587687  0.00154514 -0.02619711
 -0.03361982 -0.00701159 -0.02402627 -0.01984342 -0.02143483 -0.00908037
 -0.02334298  0.01601315  0.01011295  0.05194086  0.00670534  0.01274134
 -0.02312072  0.04286743  0.00765548 -0.03113597  0.02371593  0.03747172
  0.04521442 -0.02764486]
blood    [ 0.67308418 -0.08168938 -0.02032919 -0.03726672 -0.01857222 -0.00592257
 -0.06235834  0.17425434  0.04792914  0.06238527 -0.04550545 -0.0383702
 -0.04933893  0.0045425   0.03732097  0.03057487  0.07542497 -0.04557601
  0.01892432 -0.06496366  0.0414332   0.05367409  0.09649736 -0.00711013
  0.01733989 -0.06656684  0.14664991  0.03196268 -0.04392479  0.06460658
 -0.

amaze    [ 0.87390739  0.13544734 -0.05514322 -0.08542758 -0.04945112  0.11218594
 -0.01837037 -0.01993921 -0.01451999 -0.01285693  0.06991706  0.008284
  0.08923875  0.03662422 -0.05567769 -0.00482382  0.06125564 -0.12339505
 -0.00359242  0.02038066 -0.01361743  0.03120493 -0.04614771 -0.03496219
  0.0383586  -0.04308003  0.00598053  0.04938194  0.02407103 -0.02401902
  0.09366294 -0.01002173 -0.00489173  0.03524498 -0.02411883 -0.02072844
 -0.04566738 -0.01596795  0.02358238  0.0240492   0.00748074  0.02352196
  0.00993167  0.04157315 -0.00865395 -0.03375572 -0.02456111 -0.00236873
  0.00941059 -0.01738102]
appearance    [ 0.6374877  -0.0026045   0.07280056 -0.06420555  0.17873768 -0.18510023
  0.41348536  0.18712342 -0.06804585  0.07563638  0.32207763 -0.0573893
 -0.02448473 -0.1791139  -0.04261394 -0.0634357   0.03888806  0.06763059
 -0.02422838  0.07051847  0.06345706 -0.02246951  0.03493945 -0.02748304
  0.05628892  0.0493688   0.0784092  -0.08551638 -0.00295477 -0.04362415
  0.0

In [44]:
#მონაცემები ანუ ფილმის რევიუები გარდავქმნათ სიტყვების ემბედინგების მიმდევრობად
# იგივე ზუსტად ოღონდ ახალი ემბედინგებით
sequences = []
for review in df_labeledTrainData['review']:
    sequence = [ ( word_vectors[word_to_idx[word]]) for word in review.split() if word in word_to_idx]
    sequences.append(sequence)
    
sequences_test = []
for review in df_testData['review']:
    sequence = [ ( word_vectors[word_to_idx[word]]) for word in review.split() if word in word_to_idx]
    sequences_test.append(sequence)


In [45]:
#გრძელი რევოუები პროცესს შეანელებს ამიტომ 300 სიტყვის მეტს აღარ მივაქცენ ყურადღებას და 300 ზე გავაკეთებ პადინგს.
maxlen = 300

x = pad_sequences(sequences, maxlen=maxlen, dtype='float32', padding='post', truncating='post')
y = np.array(df_labeledTrainData['sentiment'])

test_x = pad_sequences(sequences_test, maxlen=maxlen, dtype='float32', padding='post', truncating='post')

In [46]:
print(x.shape)
print(test_x.shape)
print(df_labeledTrainData['sentiment'].shape)
#ზომები კარგად არის

(25000, 300, 50)
(25000, 300, 50)
(25000,)


In [47]:
#ვაკეთებ ასე:
#ვუშვებ ინფუთ ლეიერიდან lstm ლეიერში მერე overfit რო არ მოხდეს ვუკეთებ დროფაუთს 
#და ამას ვიმეორებ 2 ჯერ გრადიენტის ვანიშინგი რო არ მოხდეს
#მერე flatten - ს ვუკეთებ და საბოლოოდ Dense ლეიერით ერთ ნოუდში მისული რიცხვს ვდებ სიგმოიდ ფუნქვიას
model = Sequential([])
model.add(layers.Input(shape=(300,50)))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.Dropout(0.1))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.Dropout(0.1))
model.add(layers.Flatten())
model.add(layers.Dense(1, activation='sigmoid'))

In [48]:
df_labeledTrainData['sentiment'] = df_labeledTrainData['sentiment'].astype(np.float32)

#ადამის ოპტიმაიზერით ვუშვებ მოდელს 
#რადგან სენტიმენტი ან უარყოფითია ან დადებითი loss='binary_crossentropy'
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

In [49]:
y = np.array(df_labeledTrainData['sentiment'])

# ვაძლევ მოდელს x და y-ს
# ვაძლევ მოდელს ტრეინ და ტარგეტ სეტს
# ვაკეთებ validation_split=0.2, 80/20 ტრეინ და ვალიდაციის სეტს
# epochs=4 (მაინც არ ვარგა ამ მოდელის ემბედინგები და 4 - წესით ეყოფა, უკეთეს ემბედინგებს მერე გავაკეთებ)
history = model.fit(
    x=x,
    y=y,
    batch_size=25,
    epochs=4,
    verbose="auto",
    callbacks=None,
    validation_split=0.2,
    validation_data=None,
    shuffle=True,
    class_weight=None,
    sample_weight=None,
    initial_epoch=0,
    steps_per_epoch=None,
    validation_steps=None,
    validation_batch_size=None,
    validation_freq=1,
)

Epoch 1/4
800/800 ━━━━━━━━━━━━━━━━━━━━ 196s 235ms/step - accuracy: 0.6717 - loss: 0.5807 - val_accuracy: 0.7314 - val_loss: 0.5568
Epoch 2/4
800/800 ━━━━━━━━━━━━━━━━━━━━ 195s 244ms/step - accuracy: 0.7759 - loss: 0.4793 - val_accuracy: 0.7836 - val_loss: 0.4761
Epoch 3/4
800/800 ━━━━━━━━━━━━━━━━━━━━ 161s 201ms/step - accuracy: 0.7791 - loss: 0.4589 - val_accuracy: 0.7854 - val_loss: 0.4725
Epoch 4/4
800/800 ━━━━━━━━━━━━━━━━━━━━ 165s 206ms/step - accuracy: 0.7865 - loss: 0.4511 - val_accuracy: 0.7874 - val_loss: 0.4594


In [56]:
# იმავე ალგორითმა ნელა იმუშავა ამიტომაც კიდევ lstm ბლოკებს დავამატებ 
# learning rate - ს გავზრდი და batch -ს რაოდენობას შევამცირებ
# და რადგან წინა მოდელზე ოვერფიტი არ იყო dropout - ლეიერებს შევამცირებ
model = Sequential([])
model.add(layers.Input(shape=(300,50)))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.LSTM(80,return_sequences=True))
model.add(layers.Flatten())
model.add(layers.Dense(1, activation='sigmoid'))

In [57]:
df_labeledTrainData['sentiment'] = df_labeledTrainData['sentiment'].astype(np.float32)

#ადამის ოპტიმაიზერით ვუშვებ მოდელს 
#რადგან სენტიმენტი ან უარყოფითია ან დადებითი loss='binary_crossentropy'
model.compile(optimizer=Adam(learning_rate=0.002), loss='binary_crossentropy', metrics=['accuracy'])

In [58]:
y = np.array(df_labeledTrainData['sentiment'])

# ვაძლევ მოდელს x და y-ს
# ვაძლევ მოდელს ტრეინ და ტარგეტ სეტს
# ვაკეთებ validation_split=0.2, 80/20 ტრეინ და ვალიდაციის სეტს
# epochs=15 (მაინც არ ვარგა ამ მოდელის ემბედინგები და 4 - წესით ეყოფა, უკეთეს ემბედინგებს მერე გავაკეთებ)
history = model.fit(
    x=x,
    y=y,
    batch_size=50,
    epochs=12,
    verbose="auto",
    callbacks=None,
    validation_split=0.2,
    validation_data=None,
    shuffle=True,
    class_weight=None,
    sample_weight=None,
    initial_epoch=0,
    steps_per_epoch=None,
    validation_steps=None,
    validation_batch_size=None,
    validation_freq=1,
)

Epoch 1/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 280s 653ms/step - accuracy: 0.6957 - loss: 0.5647 - val_accuracy: 0.7730 - val_loss: 0.4894
Epoch 2/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 248s 621ms/step - accuracy: 0.7773 - loss: 0.4722 - val_accuracy: 0.7874 - val_loss: 0.4560
Epoch 3/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 255s 637ms/step - accuracy: 0.7937 - loss: 0.4429 - val_accuracy: 0.8054 - val_loss: 0.4291
Epoch 4/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 247s 617ms/step - accuracy: 0.8133 - loss: 0.4180 - val_accuracy: 0.8130 - val_loss: 0.4111
Epoch 5/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 250s 625ms/step - accuracy: 0.8150 - loss: 0.3982 - val_accuracy: 0.8228 - val_loss: 0.4094
Epoch 6/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 248s 619ms/step - accuracy: 0.8216 - loss: 0.3927 - val_accuracy: 0.8200 - val_loss: 0.4017
Epoch 7/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 253s 632ms/step - accuracy: 0.8303 - loss: 0.3794 - val_accuracy: 0.8258 - val_loss: 0.3867
Epoch 8/12
400/400 ━━━━━━━━━━━━━━━━━━━━ 253s 632ms/step - accuracy: 0.8315 -

In [59]:
print(history.history['loss'])
print(history.history['accuracy'])
print(history.history['val_loss'])
print(history.history['val_accuracy'])

predictions = (model.predict(test_x, batch_size=20, verbose="auto", steps=None, callbacks=None) > 0.5).astype(int)
predictions_df = pd.DataFrame(data={'id': df_testData['id'], 'sentiment': predictions.flatten()})
print(predictions_df)
predictions_df.to_csv('answersFinal.csv', index = False)
# ტესტ სეტზე რო შევამოყმე kaggle იქაც 70.51 % - იანი შედეგი მომცა
#საუკეთესო მონაცემები ცალკე ფაილშია

[0.522530198097229, 0.4654684066772461, 0.43866589665412903, 0.4128272533416748, 0.3980497717857361, 0.3883134424686432, 0.3784123361110687, 0.37037184834480286, 0.3619258403778076, 0.35371294617652893, 0.3451770842075348, 0.3371480107307434]
[0.7403002381324768, 0.7812005877494812, 0.7963994741439819, 0.8141498565673828, 0.8191497921943665, 0.8256999254226685, 0.8300988078117371, 0.8357998728752136, 0.8385494947433472, 0.8422496914863586, 0.8472492098808289, 0.8514992594718933]
[0.48936691880226135, 0.4560173749923706, 0.4290979504585266, 0.41113921999931335, 0.40937042236328125, 0.4017176330089569, 0.3866713345050812, 0.3889850974082947, 0.3912163972854614, 0.3776998519897461, 0.4412039816379547, 0.3874380886554718]
[0.7729998230934143, 0.7874000072479248, 0.8053996562957764, 0.8129997849464417, 0.8227996826171875, 0.8199999332427979, 0.8257997632026672, 0.8275997638702393, 0.8221997618675232, 0.8315998315811157, 0.8036000728607178, 0.8355997204780579]
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# კაგლზე 83.6% 